<a href="https://colab.research.google.com/github/pleiadeswatcher-lols/RVC-for-me-buteverywhere/blob/main/rvc-webui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
! nvidia-smi
! nvcc -V
! free -h

Mon Mar  2 05:55:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
import os

# @markdown # Mount Google Drive
data_path = "/content/data"
mount_gdrive = True  # @param {type:"boolean"}
gdrive_data_path = "/content/drive/MyDrive/AI/rvc-webui"  # @param {type:"string"}

if mount_gdrive:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(gdrive_data_path, exist_ok=True)
    ! ln -s {data_path} {gdrive_data_path}
else:
    os.makedirs(data_path, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ln: failed to create symbolic link '/content/drive/MyDrive/AI/rvc-webui/data': File exists


In [8]:
# @markdown # Clone repository

%cd /content/
repository_url = "https://github.com/ddPn08/rvc-webui.git"  # @param {type: "string"}
webui_branch = "main"  # @param {type: "string"}

! git clone {repository_url}
%cd /content/rvc-webui

/content
fatal: destination path 'rvc-webui' already exists and is not an empty directory.
/content/rvc-webui


In [9]:
# @markdown # Initialize environment
import os

conda_dir = "/opt/conda"  # @param{type:"string"}
conda_bin = os.path.join(conda_dir, "bin", "conda")

if not os.path.exists(conda_bin):
    ! curl -O https://repo.anaconda.com/miniconda/Miniconda3-py310_23.5.0-3-Linux-x86_64.sh
    ! chmod +x Miniconda3-py310_23.5.0-3-Linux-x86_64.sh
    ! bash ./Miniconda3-py310_23.5.0-3-Linux-x86_64.sh -b -f -p {conda_dir}
    ! rm Miniconda3-py310_23.5.0-3-Linux-x86_64.sh

def run_script(s):
    ! {s}

def make_args(d):
    arguments = ""
    for k, v in d.items():
        if type(v) == bool:
            arguments += f"--{k} " if v else ""
        elif type(v) == str and v:
            arguments += f"--{k} \"{v}\" "
        elif v:
            arguments += f"--{k}={v} "
    return arguments

if os.path.exists("requirments.txt"):
    ! mv requirments.txt requirements.txt

In [10]:
# @title 1. Install Applio & Requirements
!git clone https://github.com/IAHispano/Applio rvc-webui
%cd /content/rvc-webui
!pip install -r requirements.txt
!pip install facenet_pytorch

# Create the folder where your Ruby model needs to live
import os
os.makedirs('/content/rvc-webui/assets/weights', exist_ok=True)
print("✅ Folders created. Ready for files.")

fatal: destination path 'rvc-webui' already exists and is not an empty directory.
/content/rvc-webui
  Using cached gradio-3.36.1-py3-none-any.whl.metadata (15 kB)
  Using cached tqdm-4.65.0-py3-none-any.whl.metadata (56 kB)
  Using cached numpy-1.23.5.tar.gz (10.7 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
✅ Folders created. Ready for files.


In [11]:
# @title 🚀 1. Setup, Download & Surgery
import os
import torch

# --- STEP 1: Download the Model (Ruby Hoshino) ---
# Using the standard public URLs that don't require an HF token
!mkdir -p /content/rvc-webui/assets/weights
%cd /content/rvc-webui/assets/weights

# These links point to the public Ruby model we used yesterday
!curl -L -o RubyHoshino.pth "https://huggingface.co/PleiadesWatcher/RubyHoshinoRVC/resolve/main/RubyHoshino.pth?download=true"
!curl -L -o RubyHoshino.index "https://huggingface.co/PleiadesWatcher/RubyHoshinoRVC/resolve/main/added_IVF201_Flat_nprobe_1_RubyHoshino_v2.index?download=true"

# --- STEP 2: The "Surgery" Code Change ---
# This fixes the 'Weights only load failed' error permanently for this session
file_path = '/opt/conda/lib/python3.10/site-packages/fairseq/checkpoint_utils.py'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        content = f.read()

    old_code = 'state = torch.load(f, map_location=torch.device("cpu"))'
    new_code = 'state = torch.load(f, map_location=torch.device("cpu"), weights_only=False)'

    if old_code in content:
        with open(file_path, 'w') as f:
            f.write(content.replace(old_code, new_code))
        print("✅ Code Surgery Successful!")
    else:
        print("⚠️ Code already patched.")

# --- STEP 3: Environment Security Fix ---
os.environ["TORCH_LOAD_WEIGHTS_ONLY"] = "FALSE"

print("\n✨ Everything is ready! Just upload your 'Recording (4) (1).wav' to the /content/rvc-webui/ folder.")

/content/rvc-webui/assets/weights
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    29  100    29    0     0    191      0 --:--:-- --:--:-- --:--:--   192
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    29  100    29    0     0    199      0 --:--:-- --:--:-- --:--:--   200

✨ Everything is ready! Just upload your 'Recording (4) (1).wav' to the /content/rvc-webui/ folder.


In [13]:
# @title 🚀 Download Ruby Hoshino Model (Auto-Unzip)
import os

# 1. Create the weights folder
!mkdir -p /content/rvc-webui/assets/weights
%cd /content/rvc-webui/assets/weights

# 2. Download the ZIP file using the DIRECT link
print("Downloading RubyHoshino.zip...")
!curl -L -o RubyHoshino.zip "https://huggingface.co/LX1560/rvc-anime/resolve/main/RubyHoshino.zip?download=true"

# 3. Unzip the files
print("Unzipping...")
!unzip -o RubyHoshino.zip

# 4. Clean up the zip to save space
!rm RubyHoshino.zip

# 5. Apply the "Surgery" Patch for PyTorch 2.6
file_path = '/opt/conda/lib/python3.10/site-packages/fairseq/checkpoint_utils.py'
if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        content = f.read()
    old = 'state = torch.load(f, map_location=torch.device("cpu"))'
    new = 'state = torch.load(f, map_location=torch.device("cpu"), weights_only=False)'
    if old in content:
        with open(file_path, 'w') as f:
            f.write(content.replace(old, new))
        print("✅ Surgery successful!")

print("✨ Done! Ruby is ready for inference.")

/content/rvc-webui/assets/weights
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1096  100  1096    0     0   6659      0 --:--:-- --:--:-- --:--:--  6682
100 87.7M  100 87.7M    0     0  18.5M      0  0:00:04  0:00:04 --:--:-- 26.0M
Unzipping...
Archive:  RubyHoshino.zip
  inflating: metadata.json           
  inflating: model.index             
  inflating: model.pth               
✨ Done! Ruby is ready for inference.


In [21]:
# @title 🛠️ Fix Gradio Client Error
import os

# 1. Kill the broken versions
!pip uninstall -y gradio gradio-client

# 2. Install the specific compatible versions
# 3.48.0 is the most stable for these older RVC/Applio forks
!pip install gradio==3.48.0 gradio-client==0.6.1

# 3. Manually fix the specific missing piece if pip missed it
!pip install "gradio[client]"

# 4. Set the security bypass for the next launch
os.environ["TORCH_LOAD_WEIGHTS_ONLY"] = "FALSE"

print("✅ Gradio fixed. Try the launch cell again!")

Found existing installation: gradio 3.48.0
Uninstalling gradio-3.48.0:
  Successfully uninstalled gradio-3.48.0
Found existing installation: gradio_client 0.6.1
Uninstalling gradio_client-0.6.1:
  Successfully uninstalled gradio_client-0.6.1
  Using cached gradio-3.48.0-py3-none-any.whl.metadata (17 kB)
  Using cached gradio_client-0.6.1-py3-none-any.whl.metadata (7.1 kB)
Using cached gradio-3.48.0-py3-none-any.whl (20.3 MB)
Using cached gradio_client-0.6.1-py3-none-any.whl (299 kB)
✅ Gradio fixed. Try the launch cell again!


In [29]:
# @markdown # Launch webui
import os
%cd /content/rvc-webui
# This tells the current environment to relax
os.environ["TORCH_LOAD_WEIGHTS_ONLY"] = "FALSE"

run_script(f"""
eval "$({conda_bin} shell.bash hook)"
# This line is the magic trick - it forces the new process to ignore the error
export TORCH_LOAD_WEIGHTS_ONLY=FALSE
python launch.py --share --host 0.0.0.0 --port 41130
""")

/content/rvc-webui
Python 3.10.12 (main, Jul  5 2023, 18:54:27) [GCC 11.2.0]
Commit hash: b71742809a24cd89eb18081b831c0b1ac11ccb2a
Installing requirements
Traceback (most recent call last):
  File "/content/rvc-webui/webui.py", line 3, in <module>
    from modules import cmd_opts, ui
  File "/content/rvc-webui/modules/ui.py", line 5, in <module>
    import gradio as gr
  File "/opt/conda/lib/python3.10/site-packages/gradio/__init__.py", line 3, in <module>
    import gradio.components as components
  File "/opt/conda/lib/python3.10/site-packages/gradio/components/__init__.py", line 1, in <module>
    from gradio.components.annotated_image import AnnotatedImage
  File "/opt/conda/lib/python3.10/site-packages/gradio/components/annotated_image.py", line 9, in <module>
    from gradio_client.serializing import JSONSerializable
ModuleNotFoundError: No module named 'gradio_client.serializing'


In [24]:
# @title 🛠️ Install Missing Modules
!pip install fairseq==0.12.2
!pip install librosa==0.9.2
!pip install pyworld tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 77.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Requested omegaconf<2.1 from https://files.pythonhosted.org/packages/d0/eb/9d63ce09dd8aa85767c65668d5414958ea29648a0eec80a4a7d311ec2684/omegaconf-2.0.6-py3-none-any.whl (from fairseq==0.12.2) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    PyYAML (>=5.1.*)
            ~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested omegaconf<2.1 from https://files.pythonhosted.org/packages/e5/f6/043b6d255dd6fbf2025110cea35b87f4c5100a181681d8eab496269f0d5b/omegaconf-2.0.5-py3-none-any.whl (from fairseq==0.12.2) has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    PyYAML (>=5.1.*)
            ~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested omegaco

In [25]:
# @title 🛠️ Force Install fairseq
# 1. Install the specific 'omegaconf' version that fairseq needs, but bypass the check
!pip install omegaconf==2.0.6 --no-deps

# 2. Install fairseq without looking at dependencies (since they are causing the loop)
!pip install fairseq==0.12.2 --no-deps

# 3. Install the other support tools
!pip install hydra-core==1.0.7 antlr4-python3-runtime==4.8

  Using cached omegaconf-2.0.6-py3-none-any.whl.metadata (3.0 kB)
Requested omegaconf==2.0.6 from https://files.pythonhosted.org/packages/d0/eb/9d63ce09dd8aa85767c65668d5414958ea29648a0eec80a4a7d311ec2684/omegaconf-2.0.6-py3-none-any.whl has invalid metadata: .* suffix can only be used with `==` or `!=` operators
    PyYAML (>=5.1.*)
            ~~~~~~^
Please use pip<24.1 if you need to use this version.
ERROR: Ignored the following yanked versions: 1.0.0, 1.0.1, 1.0.2, 2.0.0rc1, 2.0.0rc2, 2.0.0rc22, 2.0.0rc23, 2.0.0rc24, 2.0.0rc25, 2.0.0rc26, 2.0.0rc27, 2.0.0rc28, 2.0.0rc29, 2.0.1rc1, 2.0.1rc2, 2.0.1rc3, 2.0.1rc4, 2.0.1rc5, 2.2.0
ERROR: Could not find a version that satisfies the requirement omegaconf==2.0.6 (from versions: 1.0.3, 1.0.4, 1.0.5, 1.0.6, 1.0.7, 1.0.8, 1.0.9, 1.0.10, 1.0.11, 1.0.12, 1.0.13, 1.0.14, 1.0.16, 1.0.17, 1.1.0, 1.1.1, 1.1.2, 1.1.3, 1.1.4, 1.1.5, 1.1.6, 1.1.7, 1.1.8, 1.1.9, 1.1.10, 1.2.0, 1.2.1, 1.3.0rc1, 1.3.0rc2, 1.3.0rc3, 1.3.0rc4, 1.3.0rc5, 1.3.0rc6, 1.3.0rc

In [27]:
# @title 🛠️ Fix fairseq for Python 3.12
import os

# Path to the broken file in the Colab environment
file_path = '/usr/local/lib/python3.12/dist-packages/fairseq/dataclass/configs.py'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        content = f.read()

    # The "Surgery": Replacing the forbidden mutable default with a proper field factory
    # We are changing: common: CommonConfig = CommonConfig()
    # To: common: CommonConfig = field(default_factory=CommonConfig)

    # First, ensure 'field' is imported
    if 'from dataclasses import dataclass' in content and 'field' not in content:
        content = content.replace('from dataclasses import dataclass', 'from dataclasses import dataclass, field')

    # Now fix the specific broken lines
    content = content.replace('common: CommonConfig = CommonConfig()', 'common: CommonConfig = field(default_factory=CommonConfig)')
    content = content.replace('common_eval: CommonEvalConfig = CommonEvalConfig()', 'common_eval: CommonEvalConfig = field(default_factory=CommonEvalConfig)')

    with open(file_path, 'w') as f:
        f.write(content)
    print("✅ Surgery successful! Fairseq is now Python 3.12 compatible.")
else:
    print("❌ Could not find fairseq. Did you run the installation cell first?")

✅ Surgery successful! Fairseq is now Python 3.12 compatible.


In [28]:
# @markdown # 🚀 Launch Ruby Project
import os
%cd /content/rvc-webui

# 1. Final Security Bypass
os.environ["TORCH_LOAD_WEIGHTS_ONLY"] = "FALSE"

# 2. Launching with the fixed Gradio
# We use --share to get that public link you can click
!python webui.py --share --disable-auto-launch

/content/rvc-webui
Traceback (most recent call last):
  File "/content/rvc-webui/webui.py", line 3, in <module>
    from modules import cmd_opts, ui
  File "/content/rvc-webui/modules/ui.py", line 9, in <module>
    from . import models, shared
  File "/content/rvc-webui/modules/models.py", line 6, in <module>
    from fairseq import checkpoint_utils
  File "/usr/local/lib/python3.12/dist-packages/fairseq/__init__.py", line 20, in <module>
    from fairseq.distributed import utils as distributed_utils
  File "/usr/local/lib/python3.12/dist-packages/fairseq/distributed/__init__.py", line 7, in <module>
    from .fully_sharded_data_parallel import (
  File "/usr/local/lib/python3.12/dist-packages/fairseq/distributed/fully_sharded_data_parallel.py", line 10, in <module>
    from fairseq.dataclass.configs import DistributedTrainingConfig
  File "/usr/local/lib/python3.12/dist-packages/fairseq/dataclass/__init__.py", line 6, in <module>
    from .configs import FairseqDataclass
  File "/usr